# Chapter 09 — One Runtime, Many Windows

**Companion to Applied AI**

Question: What breaks when state lives in the interface instead of one runtime?

By the end of this notebook you will have:

- built one SQLite event store shared by three simulated surfaces
- showed write-once / read-anywhere / resume-anywhere
- contrasted it with three independent histories diverging

## What this notebook demonstrates
A tiny runtime (SQLite event ledger) with three simulated surfaces. Uses only the standard library.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)
import sqlite3, json, time

seed: 42


## 1. One runtime: a shared event store

In [2]:
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE events(id INTEGER PRIMARY KEY, surface TEXT, kind TEXT, payload TEXT, ts REAL)")
def emit(surface: str, kind: str, payload: dict):
    db.execute("INSERT INTO events(surface, kind, payload, ts) VALUES (?,?,?,?)",
               (surface, kind, json.dumps(payload), time.time()))
    db.commit()

emit("cli", "decision", {"task": "deploy", "choice": "canary-10pct"})
emit("editor", "note", {"task": "deploy", "text": "canary looks green"})
print(db.execute("SELECT surface, kind, payload FROM events").fetchall())

[('cli', 'decision', '{"task": "deploy", "choice": "canary-10pct"}'), ('editor', 'note', '{"task": "deploy", "text": "canary looks green"}')]


## 2. Surface B reads what surface A wrote; surface C resumes the task

In [3]:
def history(task: str):
    rows = db.execute("SELECT surface, kind, payload FROM events ORDER BY id").fetchall()
    return [(s, k, json.loads(p)) for s, k, p in rows if json.loads(p).get("task") == task]

h = history("deploy")
print("mobile surface sees:", h)
assert any(k == "decision" for _, k, _ in h), "decision must be visible to every surface"

mobile surface sees: [('cli', 'decision', {'task': 'deploy', 'choice': 'canary-10pct'}), ('editor', 'note', {'task': 'deploy', 'text': 'canary looks green'})]


## 3. Contrast: three independent in-memory histories diverge

In [4]:
cli_hist, mobile_hist = [], []
cli_hist.append({"task": "deploy", "choice": "canary-10pct"})
# mobile never saw it: its operator chooses differently
mobile_hist.append({"task": "deploy", "choice": "full-rollout"})
print("CLI believes:   ", cli_hist[-1]["choice"])
print("mobile believes:", mobile_hist[-1]["choice"])
assert cli_hist[-1]["choice"] != mobile_hist[-1]["choice"]
print("Two interfaces, two processes, one incident.")

CLI believes:    canary-10pct
mobile believes: full-rollout
Two interfaces, two processes, one incident.


## Interpretation
- Supports: shared durable state makes surfaces views; interface-local state forks the process.
- Does NOT support: a production sync protocol (no conflict resolution here).

## Try it yourself
1. Add a fourth surface (`voice`) that resumes `deploy` from the ledger.
2. Delete the ledger mid-demo and show what each surface can still prove.
3. Add a `supersedes` event and compute the current choice as a fold over events.